# 06 — Semantic Embeddings & Complaint Clustering

**Project:** Customer Complaint Root-Cause Classifier  
**Module:** 5 — Unsupervised Theme Discovery and Semantic Clustering  
**Embedding Model:** `sentence-transformers/all-MiniLM-L6-v2`  
**Techniques:** UMAP Dimensionality Reduction, HDBSCAN Density Clustering, c-TF-IDF Theme Extraction  

### Objectives:
1. Generate dense semantic embeddings using **Sentence Transformers** in memory-bounded batches (tuned for 8 GB RAM).
2. Leverage narrative deduplication to avoid redundant compute on identical synthetic templates.
3. Reduce embedding dimensions using **UMAP** (5D for density clustering, 2D for interactive visualization).
4. Cluster complaints using **HDBSCAN** to discover organic complaint themes without presuming cluster counts.
5. Extract descriptive theme keywords for each cluster using **c-TF-IDF** (Class-Based TF-IDF).
6. Identify **centroid-closest representative complaints** for each cluster.
7. Visualize the cluster landscape using an interactive **Plotly scatter plot**.
8. Persist embeddings and cluster summaries to disk for downstream Root-Cause Investigation (Module 6) and the Streamlit Dashboard.

## 1. Setup and Environment

In [ ]:
import sys
import os
from pathlib import Path

# Ensure project root is on sys.path
project_root = Path(os.path.abspath("")).resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns

from config.settings import (
    PROJECT_ROOT, EMBEDDINGS_DIR, REPORTS_DIR, EMBEDDING_MODEL_NAME, RANDOM_SEED
)
from src.data.load_data import load_complaints
from src.data.clean_data import clean_complaints
from src.utils.helpers import get_memory_usage
from src.clustering.generate_embeddings import (
    load_embedding_model, generate_embeddings, generate_and_save_embeddings
)
from src.clustering.cluster_complaints import (
    reduce_dimensions_umap, compute_2d_projection, cluster_with_hdbscan, compute_clustering_metrics
)
from src.clustering.topic_extraction import (
    compute_c_tfidf, find_representative_complaints, generate_cluster_summaries
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Embedding model: {EMBEDDING_MODEL_NAME}")
print(f"Current memory usage: {get_memory_usage():.2f} MB")

## 2. Load Complaint Records

In [ ]:
raw_df = load_complaints()
df = clean_complaints(raw_df)

print(f"Total complaints loaded: {len(df):,}")
print(f"Unique complaint narratives: {df['complaint_narrative'].nunique():,}")

# For responsive interactive clustering analysis, we sample 10,000 complaints
SAMPLE_SIZE = 10000
sample_df = df.sample(n=SAMPLE_SIZE, random_state=RANDOM_SEED).reset_index(drop=True)
print(f"Working sample size: {len(sample_df):,} records")

## 3. Generate Semantic Embeddings

Using `sentence-transformers/all-MiniLM-L6-v2` (384-dimensional dense vectors) in batches of 512.

In [ ]:
# Check if precomputed embeddings exist, otherwise generate
emb_file = EMBEDDINGS_DIR / f"sample_{SAMPLE_SIZE}_embeddings.npy"

if emb_file.exists():
    print(f"Loading cached embeddings from {emb_file}...")
    embeddings = np.load(emb_file)
else:
    print(f"Generating embeddings for {len(sample_df):,} texts...")
    embeddings = generate_embeddings(
        sample_df["complaint_narrative"].tolist(),
        batch_size=512,
        show_progress=True
    )
    os.makedirs(EMBEDDINGS_DIR, exist_ok=True)
    np.save(emb_file, embeddings)
    print(f"Saved embeddings to {emb_file}")

print(f"Embeddings matrix shape: {embeddings.shape}")
print(f"Embedding dimension: {embeddings.shape[1]}")
print(f"Memory usage: {get_memory_usage():.2f} MB")

## 4. UMAP Dimensionality Reduction (5D for Clustering & 2D for Plotting)

High-dimensional density estimation suffers from distance concentration. Reducing from 384D to 5D preserves local and global topology for HDBSCAN.

In [ ]:
print("Running UMAP 5D reduction for clustering...")
reduced_5d = reduce_dimensions_umap(embeddings, n_components=5, n_neighbors=15, min_dist=0.0)

print("Running UMAP 2D projection for visualization...")
coords_2d = compute_2d_projection(embeddings, n_neighbors=15, min_dist=0.1)

sample_df["umap_x"] = coords_2d[:, 0]
sample_df["umap_y"] = coords_2d[:, 1]
print("Dimensionality reduction complete.")

## 5. HDBSCAN Density-Based Clustering

In [ ]:
# Run HDBSCAN on 5D UMAP representation
cluster_labels = cluster_with_hdbscan(
    reduced_5d,
    min_cluster_size=25,
    min_samples=10
)
sample_df["cluster_id"] = cluster_labels

# Compute diagnostics
metrics = compute_clustering_metrics(embeddings, cluster_labels)
print("\n=== CLUSTERING METRICS ===")
print(f"Total points:        {metrics['n_samples']:,}")
print(f"Discovered clusters: {metrics['n_clusters']}")
print(f"Noise points (-1):   {metrics['n_noise_points']:,} ({metrics['noise_ratio']:.1%})")
if metrics['silhouette_score'] is not None:
    print(f"Silhouette score:    {metrics['silhouette_score']:.4f}")

## 6. Extract Cluster Themes via c-TF-IDF

In [ ]:
topics_map = compute_c_tfidf(
    sample_df["complaint_narrative"].tolist(),
    cluster_labels,
    top_n_words=6
)

# Assign theme names to clusters
sample_df["cluster_theme"] = sample_df["cluster_id"].apply(
    lambda cid: ", ".join(topics_map[cid][:3]) if cid != -1 else "Outlier / Unclustered"
)

print("Top 10 Discovered Cluster Themes:")
for cid in sorted(list(topics_map.keys()))[:10]:
    count = (sample_df["cluster_id"] == cid).sum()
    keywords = ", ".join(topics_map[cid])
    print(f"  Cluster {cid:2d} ({count:4d} complaints): {keywords}")

## 7. Representative Complaints per Cluster

In [ ]:
reps_map = find_representative_complaints(sample_df, embeddings, n_samples=2)

print("=== REPRESENTATIVE COMPLAINT SAMPLES ===\n")
for cid in sorted(list(reps_map.keys()))[:5]:
    print(f"--- Cluster {cid}: {', '.join(topics_map.get(cid, []))} ---")
    for r in reps_map[cid]:
        print(f"  * [{r['complaint_id']}] (Cosine Sim: {r['similarity']:.3f}) | Issue: {r['issue']}")
        print(f"    \"{r['narrative'][:120]}...\"")
    print()

## 8. Generate Cluster Profile Summary Table

In [ ]:
summary_df = generate_cluster_summaries(sample_df, embeddings)
print("Cluster Profile Table (Top 10):")
summary_df.head(10)[["cluster_id", "theme_keywords", "complaint_count", "share_pct", "dominant_issue"]]

## 9. Interactive 2D Cluster Map (Plotly)

In [ ]:
# Truncate narrative for clean hover tooltip
sample_df["hover_text"] = sample_df["complaint_narrative"].str.slice(0, 100) + "..."

fig = px.scatter(
    sample_df[sample_df["cluster_id"] != -1],  # Plot non-noise clusters
    x="umap_x",
    y="umap_y",
    color="cluster_theme",
    hover_data={"complaint_id": True, "issue": True, "sub_issue": True, "hover_text": True},
    title=f"UMAP Semantic Cluster Landscape (HDBSCAN: {metrics['n_clusters']} Themes Discovered)",
    labels={"umap_x": "UMAP 1", "umap_y": "UMAP 2", "cluster_theme": "Discovered Theme"},
    opacity=0.7,
)
fig.update_traces(marker=dict(size=4))
fig.update_layout(height=700, legend=dict(orientation="h", y=-0.15))
fig.show()

## 10. Persist Clustered Data for Module 6 (Root Cause)

In [ ]:
clustered_csv = REPORTS_DIR / "sample_clustered_complaints.csv"
sample_df.to_csv(clustered_csv, index=False)
print(f"Saved clustered dataset to: {clustered_csv}")
print("Module 5 pipeline executed successfully!")